In [4]:
import datasets
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import os, csv

dataset = datasets.load_dataset("cyberagent/crello", revision="5.0.0")

In [5]:
import json
from PIL import Image
from io import BytesIO
import requests
import base64
import os
import re


def _serialize_value(v):
    # Handle numpy scalar types if numpy is available
    try:
        import numpy as _np
        if isinstance(v, (_np.integer, _np.floating, _np.generic, _np.bool_)):
            return v.item()
    except Exception:
        pass
    if v is None:
        return ""
    if isinstance(v, (list, dict)):
        try:
            return json.dumps(v, ensure_ascii=False)
        except Exception:
            return json.dumps([str(x) for x in v], ensure_ascii=False)
    if isinstance(v, (bytes, bytearray)):
        # keep raw bytes as base64 to avoid corruption
        try:
            import base64 as _b64
            return _b64.b64encode(v).decode('ascii')
        except Exception:
            return str(v)
    if isinstance(v, (str, int, float, bool)):
        return v
    return str(v)


def _sanitize_part(s):
    s = str(s)
    # remove path separators and odd characters
    s = s.replace(os.sep, '_').replace('/', '_').replace('\\', '_')
    # allow only a safe subset of chars
    s = re.sub(r'[^A-Za-z0-9._-]', '_', s)
    return s


def _save_single_image(item, out_dir, split, col, row_idx, elem_idx=None, elements_dir=None):
    """Try to save a single image-like object and return relative path or None."""
    # prepare dirs
    images_dir = os.path.join(out_dir, 'images')
    os.makedirs(images_dir, exist_ok=True)
    if elements_dir is None:
        elements_dir = os.path.join(out_dir, 'elements')
    os.makedirs(elements_dir, exist_ok=True)

    safe_col = _sanitize_part(col)

    # helper to sanitize extension
    def _clean_ext(ext):
        ext = str(ext or 'png')
        # drop querystrings if any
        ext = ext.split('?')[0]
        ext = ext.lstrip('./\\')
        ext = re.sub(r'[^A-Za-z0-9]', '', ext) or 'png'
        return ext

    # PIL Image
    try:
        if isinstance(item, Image.Image):
            ext = 'png'
            if elements_dir is not None and elem_idx is not None:
                fname = f"{split}_elements_{row_idx}_{elem_idx}.{_clean_ext(ext)}"
                path = os.path.join(elements_dir, fname)
                item.save(path)
                return os.path.join(os.path.basename(elements_dir), fname)
            else:
                fname = f"{split}_{safe_col}_{row_idx}.{_clean_ext(ext)}"
                path = os.path.join(images_dir, fname)
                item.save(path)
                return os.path.join('images', fname)
    except Exception:
        pass

    # bytes-like
    if isinstance(item, (bytes, bytearray)):
        try:
            img = Image.open(BytesIO(item)).convert('RGB')
            ext = 'png'
            if elements_dir is not None and elem_idx is not None:
                fname = f"{split}_elements_{row_idx}_{elem_idx}.{_clean_ext(ext)}"
                path = os.path.join(elements_dir, fname)
                img.save(path)
                return os.path.join(os.path.basename(elements_dir), fname)
            else:
                fname = f"{split}_{safe_col}_{row_idx}.{_clean_ext(ext)}"
                path = os.path.join(images_dir, fname)
                img.save(path)
                return os.path.join('images', fname)
        except Exception:
            pass

    # string: URL or local path
    if isinstance(item, str):
        s = item.strip()
        if s.lower().startswith('http'):
            try:
                resp = requests.get(s, timeout=8)
                resp.raise_for_status()
                img = Image.open(BytesIO(resp.content)).convert('RGB')
                ext = 'png'
                if elements_dir is not None and elem_idx is not None:
                    fname = f"{split}_elements_{row_idx}_{elem_idx}.{_clean_ext(ext)}"
                    path = os.path.join(elements_dir, fname)
                    img.save(path)
                    return os.path.join(os.path.basename(elements_dir), fname)
                else:
                    fname = f"{split}_{safe_col}_{row_idx}.{_clean_ext(ext)}"
                    path = os.path.join(images_dir, fname)
                    img.save(path)
                    return os.path.join('images', fname)
            except Exception:
                pass
        if os.path.exists(s):
            try:
                img = Image.open(s).convert('RGB')
                ext = os.path.splitext(s)[1].lstrip('.') or 'png'
                ext = _clean_ext(ext)
                if elements_dir is not None and elem_idx is not None:
                    fname = f"{split}_elements_{row_idx}_{elem_idx}.{ext}"
                    path = os.path.join(elements_dir, fname)
                    img.save(path)
                    return os.path.join(os.path.basename(elements_dir), fname)
                else:
                    fname = f"{split}_{safe_col}_{row_idx}.{ext}"
                    path = os.path.join(images_dir, fname)
                    img.save(path)
                    return os.path.join('images', fname)
            except Exception:
                pass
    return None


def _save_image_if_needed(v, out_dir, split, col, idx):
    """If v looks like an image (PIL Image, bytes, URL/path) or a list of such, save it and return a relative path or JSON list of paths."""
    if v is None:
        return None
    # prepare dirs
    images_dir = os.path.join(out_dir, 'images')
    elements_dir = os.path.join(out_dir, 'elements')
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(elements_dir, exist_ok=True)

    # list-like: save each element to elements_dir
    if isinstance(v, (list, tuple)):
        paths = []
        for j, item in enumerate(v, start=1):
            p = _save_single_image(item, out_dir, split, col, idx, elem_idx=j, elements_dir=elements_dir)
            if p:
                paths.append(p)
        if paths:
            try:
                return json.dumps(paths, ensure_ascii=False)
            except Exception:
                return str(paths)
        return None

    # single item: use images_dir behavior
    p = _save_single_image(v, out_dir, split, col, idx, elem_idx=None, elements_dir=None)
    return p


# Save everything under a single data folder
data_dir = os.path.join('data', 'crello')
os.makedirs(data_dir, exist_ok=True)
for split in ['train','validation']:
    data = dataset[split]
    output_dir = data_dir
    output_path = os.path.join(output_dir, f'crello_{split}.csv')
    # write rows with a progress bar to avoid building a large DataFrame in memory
    total = len(data)
    skipped = 0
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=data.column_names, quoting=csv.QUOTE_ALL)
        writer.writeheader()
        for idx, row in enumerate(tqdm(data, total=total, desc=f'Saving {split}')):
            clean_row = {}
            # Decide whether to skip this row entirely: skip if 'vk' appears in keywords
            save_images_flag = True
            kw = None
            try:
                kw = row.get('keywords') if isinstance(row, dict) else None
            except Exception:
                try:
                    kw = row['keywords'] if 'keywords' in row else None
                except Exception:
                    kw = None
            if kw:
                try:
                    # kw may be a JSON list stored as string, or a plain string
                    if isinstance(kw, str) and kw.strip().startswith('['):
                        kws = json.loads(kw)
                    elif isinstance(kw, (list, tuple)):
                        kws = kw
                    elif isinstance(kw, str):
                        kws = [x.strip() for x in kw.split(',') if x.strip()]
                    else:
                        kws = [str(kw)]
                except Exception:
                    kws = [str(kw)]
                # if any keyword equals or contains 'vk' (case-insensitive), skip the entire row
                for item in kws:
                    try:
                        if 'vk' in str(item).lower():
                            save_images_flag = False
                            break
                    except Exception:
                        continue

            if not save_images_flag:
                skipped += 1
                continue

            for k, v in row.items():
                img_path = _save_image_if_needed(v, output_dir, split, k, idx)
                if img_path is not None:
                    clean_row[k] = img_path
                else:
                    clean_row[k] = _serialize_value(v)
            writer.writerow(clean_row)
    print(f'Saved {split} split to {output_path} (skipped {skipped} rows due to keywords containing "vk")')


Saving train:   0%|          | 0/19372 [00:00<?, ?it/s]

Saved train split to data/crello/crello_train.csv (skipped 429 rows due to keywords containing "vk")


Saving validation:   0%|          | 0/1823 [00:00<?, ?it/s]

Saved validation split to data/crello/crello_validation.csv (skipped 42 rows due to keywords containing "vk")


In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Prefer the exported data/crello CSV when available
preferred = os.path.join('..', 'data', 'crello', 'crello_test.csv')
fallbacks = [
    '../data/crello_test.csv',
    'data/crello/crello_test.csv',
    'data/crello/crello_test.csv',
    'data/crello/crello_test.csv',
    'crello_test.csv'
]

csv_path = None
if os.path.exists(preferred):
    csv_path = preferred
else:
    for p in fallbacks:
        if os.path.exists(p):
            csv_path = p
            break

if csv_path is None:
    print('File not found')
else:
    print('Loading CSV from:', csv_path)
    df = pd.read_csv(csv_path)
    print('Loaded', len(df), 'rows')
    display(df.head())


Loading CSV from: data/crello/crello_test.csv
Loaded 2060 rows


,id,length,group,format,canvas_width,canvas_height,category,title,suitability,keywords,...,font_size,font_bold,font_italic,text_line,text_color,text_align,capitalize,line_height,letter_spacing,cluster_index
0,5a217b64d8141396fe9a2d7d,8,0,2,940,788,0,Thanksgiving Dinner with Roasted Turkey,[0],"[""dinner"", ""festive"", ""food"", ""hot"", ""thanksgi...",...,"[0.0, 0.0, 0.0, 0.0, 46.52853012084961, 17.0, ...","[[], [], [], [], [false, false, false, false, ...","[[], [], [], [], [false, false, false, false, ...","[[], [], [], [], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[[], [], [], [], [""rgba(255, 255, 255, 1)"", ""r...","[0, 0, 0, 0, 2, 2, 2, 2]","[false, false, false, false, false, false, tru...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]",307
1,5ea84494499b85dcc7eaa006,19,0,0,1080,1920,7,Weekly To-do list with office doodles,[0],"[""coffee"", ""office"", ""pencil"", ""work"", ""educat...",...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[], [], [], [], [], [], [], [], [], [], [fals...","[[], [], [], [], [], [], [], [], [], [], [fals...","[[], [], [], [], [], [], [], [], [], [], [0, 0...","[[], [], [], [], [], [], [], [], [], [], [""rgb...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, ...","[false, false, false, false, false, false, fal...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",162
2,5888d75195a7a863ddcc2902,10,1,16,600,200,12,Citation about Fighting poverty for a brighter...,[0],"[""charity"", ""day"", ""caucasian"", ""charity"", ""ch...",...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 28.0, 0.0,...","[[], [], [], [], [], [], [], [true, true, true...","[[], [], [], [], [], [], [], [false, false, fa...","[[], [], [], [], [], [], [], [0, 0, 0, 0, 0, 0...","[[], [], [], [], [], [], [], [""rgba(35, 31, 32...","[0, 0, 0, 0, 0, 0, 0, 3, 0, 0]","[false, false, false, false, false, false, fal...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, ...",307
3,5f0415c7499b85dcc7a2da2c,31,4,45,595,842,3,Weekly Schedule with Girl and Globe,[],"[""weekly"", ""girl"", ""globe"", ""apple"", ""yellow"",...",...,"[0.0, 0.0, 0.0, 0.0, 42.0, 14.0, 14.0, 14.0, 1...","[[], [], [], [], [false, false, false, false, ...","[[], [], [], [], [false, false, false, false, ...","[[], [], [], [], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[[], [], [], [], [""rgba(117, 64, 144, 1)"", ""rg...","[0, 0, 0, 0, 2, 1, 2, 2, 2, 2, 2, 2, 0, 0, 0, ...","[false, false, false, false, false, false, fal...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",163
4,58b436c495a7a863ddcc9f39,13,3,24,160,600,4,Design camp in London,[],"[""design"", ""designer"", ""architecture"", ""camp"",...",...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[[], [], [], [], [], [], [], [], [], [], [fals...","[[], [], [], [], [], [], [], [], [], [], [fals...","[[], [], [], [], [], [], [], [], [], [], [0, 0...","[[], [], [], [], [], [], [], [], [], [], [""rgb...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 3]","[false, false, false, false, false, false, fal...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",118
